In [ ]:
import os

import numpy as np
np.random.seed(1)
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

from sklearn.metrics import roc_curve, roc_auc_score, auc, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split, StratifiedKFold

tf.random.set_seed(1)
print(tf.__version__)
plt.style.use('ggplot')

from config.config import *

In [ ]:
# Output path
OUTPUT_PATH = '.'

In [ ]:
def cross_validation_plot_roc_curve(ground_truth, predictions, filename=None):
    plt.figure(figsize=(15,10))
    i = 0
    mean_fpr = np.linspace(0, 1, 100)
    tprs = []
    aucs = []
    for g, p in zip(ground_truth, predictions):
        # Compute ROC curve and area the curve
        fpr, tpr, thresholds = roc_curve(y_true = g, y_score = p, drop_intermediate=False)
        tprs.append(np.interp(mean_fpr, fpr, tpr))
        tprs[-1][0] = 0.0
        roc_auc = auc(fpr, tpr)
        aucs.append(roc_auc)
        plt.plot(fpr, tpr, lw=1, alpha=0.3,
                 label='ROC fold %d (AUC = %0.4f)' % (i, roc_auc))
        i += 1
    
    plt.plot([0, 1], [0, 1], linestyle='--', lw=2, color='r',
         label='Chance', alpha=.8)

    mean_tpr = np.mean(tprs, axis=0)
    mean_tpr[-1] = 1.0
    mean_auc = auc(mean_fpr, mean_tpr)
    std_auc = np.std(aucs)
    plt.plot(mean_fpr, mean_tpr, color='b',
             label=r'Mean ROC (AUC = %0.4f $\pm$ %0.4f)' % (mean_auc, std_auc),
             lw=2, alpha=.8)

    std_tpr = np.std(tprs, axis=0)
    tprs_upper = np.minimum(mean_tpr + std_tpr, 1)
    tprs_lower = np.maximum(mean_tpr - std_tpr, 0)
    plt.fill_between(mean_fpr, tprs_lower, tprs_upper, color='grey', alpha=.2,
                     label=r'$\pm$ 1 std. dev.')

    plt.xlim([-0.05, 1.05])
    plt.ylim([-0.05, 1.05])
    plt.xlabel('1 - Specificity')
    plt.ylabel('Sensitivity')
    plt.title('Receiver operating characteristic curve')
    plt.legend(loc="lower right")
    if filename:
        plt.savefig(filename, dpi=500)
    plt.show()
    return aucs

In [ ]:
notes_filename = 'MIMICIII_dataset_medical_notes.pickle'
notes = pd.read_pickle(notes_filename);
notes.sort_values(by=['subject_id', 'icu_time_hr'], ascending=[True, True], inplace=True)
print(notes['subject_id'].nunique())
notes.head(10)

In [ ]:
# Filter out test-set patients
test_icustays = pd.read_csv(I_TEST_PATH)
test_icustay_ids = test_icustays['icustay_id'].unique()

# Find subject_ids associated with test icustay_ids
test_subject_ids = notes.loc[
    notes['icustay_id'].isin(test_icustay_ids), 'subject_id'
].unique()

# Remove all rows for those subject_ids (covering future ICU stays too)
notes = notes[~notes['subject_id'].isin(test_subject_ids)]
print(f"Removed {len(test_subject_ids)} subjects ({len(test_icustay_ids)} test icustays). "
      f"Remaining unique subjects: {notes['subject_id'].nunique()}")

In [ ]:
sns.catplot(y='category', kind='count', palette="tab20", data=notes, height=10, aspect=1.3)
plt.tight_layout()
print(notes['category'].value_counts())

In [ ]:
filtered_notes = notes[notes.category.isin(['Nursing/other', 'Radiology', 'Nursing', 'Physician'])]#.sample(frac=.10, random_state=42)
filtered_notes['category'].value_counts()

In [ ]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
stops = set(stopwords.words("english"))
f = lambda x: ' '.join([item for item in x.split() if item not in stops])
filtered_notes['text'] = filtered_notes['text'].apply(f)

In [11]:
mortality = filtered_notes[['subject_id','mort_icu']].drop_duplicates(subset=['subject_id'])[['mort_icu']].values

In [ ]:
g = sns.catplot(y='category', hue='mort_icu', kind='count', data=filtered_notes, height=10, aspect=1.3)

In [13]:
pivoted_notes = filtered_notes.pivot_table(index=['subject_id', 'icu_time_hr'], columns='category', values='text', aggfunc=lambda t : '\n'.join(t))
pivoted_notes.loc[pivoted_notes['Nursing/other'] == '','Nursing/other'] = pivoted_notes[pivoted_notes['Nursing/other'] == '']['Nursing']
pivoted_notes.fillna('', inplace=True)

In [ ]:
print(len(pivoted_notes))
pivoted_notes.head(20)

In [ ]:
print(pivoted_notes.loc[3].iloc[0]['Radiology'])

In [ ]:
pivoted_notes_v2 = filtered_notes.pivot_table(index='subject_id', columns='category', values='text', aggfunc=lambda t : '\n'.join(t))
pivoted_notes_v2.loc[pivoted_notes_v2['Nursing/other'].isnull(),'Nursing/other'] = pivoted_notes_v2[pivoted_notes_v2['Nursing/other'].isnull()]['Nursing']

pivoted_notes_v2.dropna(axis=0, subset=['Nursing/other'], inplace=True)
print(len(pivoted_notes_v2))
pivoted_notes_v2.head(20)

In [ ]:
print(pivoted_notes_v2.loc[3]['Nursing/other'])

In [ ]:
mortality = filtered_notes[['subject_id','mort_icu']].drop_duplicates(subset=['subject_id'])
mortality = mortality[mortality.subject_id.isin(pivoted_notes_v2.index.values)][['mort_icu']].values
mortality.shape

In [ ]:
vocab_size = 100000
note_category = 'Nursing/other'
vocab = np.unique(np.concatenate(pivoted_notes_v2[[note_category]].values))
print(vocab.shape)
oov_tok = "<OOV>"

tokenizer = Tokenizer(num_words=vocab_size, oov_token=oov_tok)
tokenizer.fit_on_texts(vocab)
notes_sequences = tokenizer.texts_to_sequences(pivoted_notes_v2[note_category].values)


In [ ]:
import pickle
with open(os.path.join(OUTPUT_PATH, 'res', 'tokenizer.pkl'), 'wb') as f:
    pickle.dump(tokenizer, f)

In [ ]:
plt.figure(figsize=(15,10))
sns.boxplot([len(x) for x in notes_sequences])

In [ ]:
sequence_max_length = 500
padded_notes_sequences = pad_sequences(notes_sequences, sequence_max_length, truncating='pre')
print(f'Total words: {len(tokenizer.word_index) + 1}')

In [22]:
embedding_dimension = 10

def create_model(output_bias=None):

  if output_bias is not None:
    output_bias = tf.keras.initializers.Constant(output_bias)

  notes_shape = (sequence_max_length, )
  notes_input = tf.keras.layers.Input(shape=notes_shape, name='notes')
  dense_vectors = tf.keras.layers.Embedding(input_dim=vocab_size + 1, 
                                            output_dim=embedding_dimension, 
                                            input_length=sequence_max_length)(notes_input)

  notes_embeddings = tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(100,
                                                                        dropout=0.5,
                                                                        return_sequences=True),
                                                    name='bidirectional_lstm')(dense_vectors)
  notes_embeddings = tf.keras.layers.LSTM(100, dropout=0.5, name='lstm')(notes_embeddings)

  notes_embeddings = tf.keras.layers.Dropout(0.5, name = 'dropout')(notes_embeddings)
  fc1 = tf.keras.layers.Dense(50, 
                              activation='relu', 
                              kernel_initializer='he_normal',
                              name='fc1')(notes_embeddings)

  mortality = tf.keras.layers.Dense(1, 
                                    activation='sigmoid',
                                    kernel_initializer='he_normal',
                                    bias_initializer=output_bias,
                                    name='mortality')(fc1)

  return tf.keras.Model(inputs=notes_input, outputs=mortality)


def create_model_conv(output_bias=None):
  if output_bias is not None:
    output_bias = tf.keras.initializers.Constant(output_bias)
  notes_shape = (sequence_max_length, )
  notes_input = tf.keras.layers.Input(shape=notes_shape, name='notes')

  dense_vectors = tf.keras.layers.Embedding(input_dim=vocab_size + 1, 
                                            output_dim=embedding_dimension, 
                                            input_length=sequence_max_length)(notes_input)
  
  dense_vectors = tf.keras.backend.expand_dims(dense_vectors, axis=3)

  notes_embeddings = tf.keras.layers.Conv2D(32, (5, embedding_dimension), activation='relu', padding='same')(dense_vectors)

  notes_embeddings = tf.keras.layers.MaxPool2D(pool_size = (1,3), padding = 'same')(notes_embeddings)

  notes_embeddings = tf.keras.layers.Flatten()(notes_embeddings)

  fc1 = tf.keras.layers.Dense(50, 
                              activation='relu', 
                              kernel_initializer='he_normal',
                              name='fc1')(notes_embeddings)

  mortality = tf.keras.layers.Dense(1, 
                                    activation='sigmoid',
                                    kernel_initializer='he_normal',
                                    bias_initializer=output_bias,
                                    name='mortality')(fc1)

  return tf.keras.Model(inputs=notes_input, outputs=mortality)

In [ ]:
import re
import os
import shutil

def get_best_val_model(fold):
    models_dir = os.path.join(OUTPUT_PATH, 'models')
    filenames = [f for f in os.listdir(models_dir) 
                 if f.startswith(f"kfold{fold}_") and f.endswith(".h5")]
    if not filenames:
        raise FileNotFoundError(f"No .h5 files found for fold {fold} in {models_dir}")

    metric_values_str = [re.findall('0.[0-9]+', f) for f in filenames]
    best_model_filename = filenames[np.argmax([get_F1(float(m[0]), float(m[1])) for m in metric_values_str])]
    
    best_model_path = os.path.join(models_dir, best_model_filename)
    print(best_model_path)

    dest_path = os.path.join(models_dir, f'kfold{fold}_best_F1.h5')
    shutil.copy(best_model_path, dest_path)
    
    return tf.keras.models.load_model(best_model_path)

def get_F1(precision, recall):
  try:
    return 2 * (precision * recall) / (precision + recall)
  except ZeroDivisionError:
    return 0

In [24]:
X = padded_notes_sequences
y = mortality

cv = StratifiedKFold(n_splits=5, random_state=42, shuffle=True)
cv_idx = list(cv.split(X=np.zeros(X.shape[0]), y=y))

In [ ]:
import pickle
from sklearn.model_selection import train_test_split, StratifiedKFold

total = y.shape[0]
pos = y.sum()
neg = total - pos

num_epochs = 3
batch_size = 128
weight_for_0 = (1 / neg)*(total)/2.0 
weight_for_1 = (1 / pos)*(total)/2.0

class_weight = {0: weight_for_0, 1: weight_for_1}

print('Weight for class 0: {:.2f}'.format(weight_for_0))
print('Weight for class 1: {:.2f}'.format(weight_for_1))

predictions = []
predictions_training_set = []
ground_truth = []
ground_truth_training_set = []
fold_history = []

initial_bias = np.log([pos/neg])

i = 0
for train, validation in cv_idx:
    tf.keras.backend.clear_session()
    ground_truth_training_set.append(y[train])
    ground_truth.append(y[validation])
    print(f'Fold {i}. Positive examples in validation: {np.sum(y[validation])}...')
    model = create_model_conv(output_bias=initial_bias)
    model.compile(optimizer = 'adam', 
                  loss = 'binary_crossentropy', 
                  metrics=[tf.keras.metrics.Recall(), tf.keras.metrics.Precision()])
    
    checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(filepath=os.path.join(OUTPUT_PATH, 'models', f"kfold{i}_" + "precision_{val_precision:.5f}_recall_{val_recall:.5f}_loss_{val_loss:.5f}.h5"),
                                            monitor='val_loss')
    fold_history.append(model.fit(x = X[train], y = y[train], 
                                  epochs = num_epochs, batch_size = batch_size, 
                                  class_weight=class_weight,
                                  callbacks=[checkpoint_cb],
                                  validation_data=(X[validation], 
                                                   y[validation])))
    
    model = get_best_val_model(i)
    predictions_training_set.append(model.predict(X[train]))
    predictions.append(model.predict(X[validation]))
    i += 1


In [ ]:
def plot_history(metric='accuracy'):
  fig, ax = plt.subplots(2, 3, sharex='col', sharey='row', figsize=(20,15))
  for k, history in enumerate(fold_history):
      f = np.argmax
      if metric == 'loss':
        f = np.argmin
      j = f(history.history[f'val_{metric}'])
      
      ax[int(k // 3), k % 3].plot(history.history[metric])
      ax[int(k // 3), k % 3].plot(history.history[f'val_{metric}'])
      ax[int(k // 3), k % 3].set_title('{3} fold {0} (best={1:.3f}, epoch {2})'.format(k, history.history[f'val_{metric}'][j], j, metric))
      ax[int(k // 3), k % 3].set_ylabel(metric)
      ax[int(k // 3), k % 3].set_xlabel('epoch')
      ax[int(k // 3), k % 3].legend(['train', 'validation'], loc='upper left')
  plt.show()
  
plot_history('recall')
plot_history('loss')

In [ ]:
_ = cross_validation_plot_roc_curve(ground_truth_training_set, predictions_training_set,
                               filename='roc_curve_training.png')

In [ ]:
roc_aucs = cross_validation_plot_roc_curve(ground_truth, predictions, 
                                filename='roc_curve_validation.png')

In [ ]:
for i, auc_val in enumerate(roc_aucs):
    print(f"Fold {i}: Validation AUC = {auc_val:.4f}")
best_fold = int(np.argmax(roc_aucs))
print(f"\nBest fold: {best_fold} (AUC = {roc_aucs[best_fold]:.4f})")
print(f"Best model file: kfold{best_fold}_best_F1.h5")